
---

### Declaration of Usage of Generative AI
In this work, Generative AI was used to post-edit the wording of **some** written paragraphs and comments, including the ones in the final report, to improve their clarity and style, as English is not the native language of any of the group members. The underlying ideas and content remain entirely our own. In terms of coding, AI assistance was used **solely** for debugging purposes when exceptions occurred in the code. **No** generative AI tools were used as a learning assistant during the completion of this exercise.

---



****ZERO shot NER and anonymization with GLiNER****

We will use a ***[fine-tuned version of GLiNER](https://huggingface.co/urchade/gliner_multi_pii-v1)***, specialized in personally identifiable information (PII).

⚡ GOAL: Create our own small dataset with various PII mentions, and anonymize it with GLiNER!

# Load the model

In [ ]:
!pip install gliner

from gliner import GLiNER

# NOTE: No need to load the model on GPU for our small dataset
model = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
# Code copy-pasted from the model card
text = """
Harilala Rasoanaivo, un homme d'affaires local d'Antananarivo, a enregistré une nouvelle société nommée "Rasoanaivo Enterprises" au Lot II M 92 Antohomadinika. Son numéro est le +261 32 22 345 67, et son adresse électronique est harilala.rasoanaivo@telma.mg. Il a fourni son numéro de sécu 501-02-1234 pour l'enregistrement.
"""

labels = ["work", "booking number", "personally identifiable information", "driver licence", "person", "book", "full address", "company", "actor", "character", "email", "passport number", "Social Security Number", "phone number"]
entities = model.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Harilala Rasoanaivo => person
Rasoanaivo Enterprises => company
Lot II M 92 Antohomadinika => full address
+261 32 22 345 67 => phone number
harilala.rasoanaivo@telma.mg => email
501-02-1234 => Social Security Number


# Anonymize entities

In [ ]:
def anonymize_entities(text: str,
                       entities: list[dict]) -> str:
    """Anonymize entities in text by replacing them with tags like <PERSON>, <IBAN> etc.

    Args:
        text (str): The original text
        entities (list): List of entity dictionaries with 'start', 'end', and 'label' keys

    Returns:
        str: Text with entities replaced by the corresponding tags.
    """

    if not entities:
        return text

    # Convert text to a list for character-wise replacement
    text_chars = list(text)

    # Sort entities by start index descending to avoid messing up indices while replacing
    entities_sorted = sorted(entities,
                             key=lambda e: e['start'],
                             reverse=True)

    for entity in entities_sorted:
      label = entity.get('label')
      start = entity.get('start')
      end = entity.get('end')

      # Basic validation
      if (label is None
          or start is None
          or end is None
          or start < 0
          or end > len(text_chars)
          or start >= end):
            continue

      # Add '<' and '>' for the tag of the label and make the label to be uppercase and replace '_' with space
      tag = '<' + label.upper().replace('_', ' ') + '>'

      # Remove characters in the span and insert the tag at the start position
      text_chars[start:end] = [tag] + [''] * (end - start - 1)

    # Join back into string
    return ''.join(text_chars)

In [ ]:
# Example text used to test whether the anonymization function works correctly
text = (
    "Simon Clematide ist Wissenschaftlicher Mitarbeiter am Institut für Computerlinguistik der Universität Zürich in der Schweiz."
    "Seine E-Mail-Adresse lautet simon.clematide@cl.uzh.ch und er lehrt Maschinelles Lernen für die Verarbeitung natürlicher Sprache."
)

# Entity types we want the model to detect in the text
labels = ["person", "university", "email_address"]

# Use the GLiNER model to perform NER on the text given the target labels
# This returns a list of detected entities with their spans and labels
entities = model.predict_entities(text, labels)

# Replace all detected entities in the text with anonymized tags
anonymized_text = anonymize_entities(text, entities)

print(anonymized_text)

<PERSON> ist Wissenschaftlicher Mitarbeiter am Institut für Computerlinguistik der <UNIVERSITY> in der Schweiz.Seine E-Mail-Adresse lautet <EMAIL ADDRESS> und er lehrt Maschinelles Lernen für die Verarbeitung natürlicher Sprache.


# Dataset

Time to create our own, diverse dataset. Make sure to not make it easy for the model (for example, do not mention the label in the sentence, but make sure it can be inferred from the context).

In [ ]:
# 15 example sentences containing different types of PII
pii_texts = ["Der Bericht von Lukas Meier (geb. 14.06.1998) wurde gestern in der Übung zur Maschinellen Sprachverarbeitung eingereicht.",
             "Kontaktieren Sie bitte sarah.wenger@uzh.ch oder telefonisch unter 044 633 22 11, falls Sie Fragen zum Projekt zur Textklassifikation haben.",
             "Die Präsentation von Prof. Dr. Stefan Baumgartner findet morgen im Raum AND-2-03 am Institut für Informatik statt. Er ist erreichbar unter stefan.baumgartner@uzh.ch.",
             "Am schwarzen Brett hängt eine Liste mit den Matrikelnummern 22-948-210 und 22-953-444 sowie den Geburtsdaten 03.09.2000 und 11.12.1999.",
             "Johanna Keller, geboren am 12.10.1998, hat ihre Hausarbeit zum Thema Sprachmodelle in der Cloud am 12. Oktober abgegeben.",
             "Für das Tutorium am Freitag sollen Sie bitte 079 421 89 34 anrufen oder an paul.schneider@uzh.ch schreiben, um die Teilnahme zu bestätigen.",
             "Die Ergebnisse der Gruppenarbeit von Ali Mansouri und Lea Graf wurden am 15.11.2025 veröffentlicht.",
             "Das Feedback von hanna.mueller@students.uzh.ch wurde von der Assistenz akzeptiert. Sie wurde am 09.03.2001 geboren.",
             "Im Herbstsemester 2025 ist das Kolloquium zur Maschinellen Lernverfahren im Raum BIN-1-102 angesetzt. Kontaktperson: felix.rapp@uzh.ch.",
             "Der Masterstudent Tobias Furrer (matr. 23-912-876) hat seinen Bericht in der Andreasstrasse 15 eingereicht. Geburtsdatum: 21.07.1999.",
             "Falls Unterlagen fehlen, wenden Sie sich bitte an 044 634 24 78 im Sekretariat der Informatik oder sekretariat.ifi@uzh.ch.",
             "Die Datei mit den Ergebnissen wurde von marta.steiner@ifi.uzh.ch am 10.10.2025 übermittelt.",
             "Die Besprechung mit der Betreuerin Dr. Karin Schmid findet im Büro AFL-G-112 statt. Ihre Durchwahl ist 044 632 88 12.",
             "Bitte überweisen Sie die Kursgebühr von 120 CHF auf das Konto CH39 0023 0235 2332 4401 Q bis zum 05.11.2025.",
             "Im Semesterprojekt arbeitet Jonas Frei (geb. 23.01.1997) mit der Firma DataSense AG an einem Modell zur Sentiment-Analyse. Seine E-Mail lautet jonas.frei@students.uzh.ch."]

# Target PII categories that the model should detect in the texts
labels = ["person",
          "email",
          "phone number",
          "student_id",
          "date of birth",
          "submission date",
          "room number",
          "company",
          "iban"]

In [ ]:
# Print the number of texts to make sure it exceeds 10
print(f'The number of texts: {len(pii_texts)}')

The number of texts: 15


In [ ]:
# Create a dataset
dataset: list[dict] = []

for text in pii_texts:

    # Predict PII entities in the current text
    entities = model.predict_entities(text, labels)

    # Anonymize the detected entities in the text
    anonymized = anonymize_entities(text, entities)

    # Store original text, model output, and anonymized version
    dataset.append({"text": text,
                    "entities": entities,
                    "anonymized_text": anonymized})

Time to see the results!

In [ ]:
for i, example in enumerate(dataset):
    print(f"Original Text No. {i + 1}:\n", example["text"])
    print(f"Anonymized Text No. {i + 1}:\n", example["anonymized_text"])
    print('-' * 50, '\n')

Original Text No. 1:
 Der Bericht von Lukas Meier (geb. 14.06.1998) wurde gestern in der Übung zur Maschinellen Sprachverarbeitung eingereicht.
Anonymized Text No. 1:
 Der Bericht von <PERSON> (geb. <DATE OF BIRTH>) wurde <SUBMISSION DATE> in der Übung zur Maschinellen Sprachverarbeitung eingereicht.
-------------------------------------------------- 

Original Text No. 2:
 Kontaktieren Sie bitte sarah.wenger@uzh.ch oder telefonisch unter 044 633 22 11, falls Sie Fragen zum Projekt zur Textklassifikation haben.
Anonymized Text No. 2:
 Kontaktieren Sie bitte <EMAIL> oder telefonisch unter <PHONE NUMBER>, falls Sie Fragen zum Projekt zur Textklassifikation haben.
-------------------------------------------------- 

Original Text No. 3:
 Die Präsentation von Prof. Dr. Stefan Baumgartner findet morgen im Raum AND-2-03 am Institut für Informatik statt. Er ist erreichbar unter stefan.baumgartner@uzh.ch.
Anonymized Text No. 3:
 Die Präsentation von <PERSON> findet morgen im Raum <ROOM NUMBER>

# Report

### Abstract
This study explores the implementation of NER and anonymization system using the GLiNER model, applied to texts written in German. The goal was to detect personally identifiable information (PII) such as names, emails, phone numbers, and student identifiers without prior finetuning for the specific dataset.

### 1. Introduction
NER is a key task in NLP used to identify entities such as people, organizations, and locations in text. However, training robust NER systems typically requires large, annotated datasets.
This experiment uses GLiNER, a zero-shot NER model capable of recognizing entities without additional training, making it ideal for languages or domains with limited labeled data. We have set the focus on German-language text in an academic context referring student information and university.

### 2. Methodology
### 2.1 GLiNER model

GLiNER (Generalist Language-Independent Named Entity Recognizer) enables entity detection using user-defined labels. It performs zero-shot inference, meaning it generalizes from provided label definitions without requiring training on domain-specific examples.

### 2.2 Anonymization Function
An anonymization function was implemented to systematically replace detected entities with standardized placeholders. To maintain index integrity, entities were sorted in descending order before replacement.

### 2.3 Dataset Creation
We have created a dataset of 15 German sentences, simulating communication that will be used at University of Zurich. We also defined 9 PII categories, which serve as the target entity types that GLiNER should detect when processing each sentence.

### 3 Implementation
The workflow consisted of:
1. Providing German-language sentences containing PII
2. Using GLiNER's zero-shot model to detect relevant entities based on our defined labels
3. Replacing the identified sensitive text spans with generic tags
4. Outputting the anonymized text

### 4 Discussion
The combination of GLiNER's zero-shot capabilities and our anonymization function proved effective for protecting personal data in German academic texts. Creating a small custom dataset allowed us to test a wide range of PII categories in realistic university-related scenarios. Overall, the approach was flexible, privacy-preserving, and well suited for anonymizing student and administrative communication without requiring any model retraining.

## Answers to the Questions

📝❓Discuss the benefits of using GLiNER vs a more traditional NER solution.

✅ GLiNER provides several advantages over traditional NER systems. Being **zero-shot** (able to recognize labels it was never explicitly trained on), **multilingual** (usable across a wide range of languages), and **label-flexible** (the user can define any entity types at prediction time), it avoids the need for task-specific training data and model retraining. This makes it especially appealing for data anonymization and other low-resource scenarios, such as detecting personal information in academic texts. When quick deployment, flexibility, and broad language coverage matter more than achieving the absolute highest accuracy, GLiNER offers a practical and efficient alternative to traditional NER models.

📝❓Did you encounter false positives/negatives? Discuss.

✅ **Yes**, the output contains both false positives and false negatives.
For example, in PII text 8 the model incorrectly labeled the name portion of the email address `hanna.mueller@students.uzh.ch` as a person, while the rest of the email remained untouched. This shows a **false positive** where a composite entity is split and only part of it is recognized correctly. A **false negative** appears in PII text 14, where the IBAN was not detected at all and therefore remained unchanged in the anonymized text. Overall, GLiNER performs reasonably well, but it still struggles with more complex or mixed entities such as email addresses or IBANs, and would likely benefit from finetuning or additional task-specific training data to improve its accuracy.